# 02. Preprocesado y feature engineering


Aquí se justifican las decisiones implementadas en `src/preprocessing.py`. El código del notebook llama a ese módulo para que todo el grupo use exactamente el mismo preprocesado.

In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete src/ desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from src.config import DATA_RAW, TARGET, CATEGORICAL, NUMERIC
sns.set_theme(style="whitegrid")
from src.preprocessing import add_features, build_preprocessor, split_xy, NUEVAS_NUMERICAS

In [ ]:
train = pd.read_csv(DATA_RAW / "train.csv")
X, y = split_xy(train)
X_fe = add_features(X)
X_fe[NUEVAS_NUMERICAS].describe().T

## 1. Variables nuevas

| Variable | Idea |
|---|---|
| `log_weight`, `log_reviews`, `log_shipping` | Quitar la cola larga |
| `tier_num` | `brand_tier` como número ordenado |
| `margen_neto` | margen menos descuento |
| `factor_descuento`, `log_factor_descuento` | el descuento multiplica el precio por (1 - discount) |
| `log_factor_margen` | el margen multiplica el precio por (1 + margin) |
| `envio_por_kg` | coste de envío relativo al peso |
| `material_missing` | indicador de valor perdido |

Si el precio se construye multiplicando factores, en escala logarítmica esos factores se suman, y un modelo lineal sobre `log(price)` los aprende fácilmente.

In [ ]:
nuevas = X_fe[NUEVAS_NUMERICAS].assign(log_price=np.log(y))
nuevas.corr()["log_price"].drop("log_price").sort_values()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(x=X_fe["weight_kg"], y=np.log(y), s=8, alpha=0.3, ax=ax[0])
sns.scatterplot(x=X_fe["log_weight"], y=np.log(y), s=8, alpha=0.3, ax=ax[1])
ax[0].set_ylabel("log(price)"); plt.show()

Con `log_weight` la relación con `log(price)` es mucho más lineal.

## 2. Imputación, escalado y codificación

`build_preprocessor()` crea un `ColumnTransformer`:
- **Numéricas:** imputación por mediana (por seguridad, ahora no hay huecos) y `StandardScaler` (solo para modelos lineales).
- **Categóricas:** los nulos de `material` pasan a `Desconocido`; después `OneHotEncoder` para modelos lineales u `OrdinalEncoder` para árboles.

Todo va dentro de un `Pipeline`, así en cada fold de la CV el preprocesado se ajusta solo con los datos de entrenamiento (sin fuga de información).

In [ ]:
prep = build_preprocessor()
Xt = prep.fit_transform(X_fe)
print("Forma tras preprocesar (lineal):", Xt.shape)
prep_arbol = build_preprocessor(escalar=False, onehot=False)
print("Forma tras preprocesar (árboles):", prep_arbol.fit_transform(X_fe).shape)

## 3. Comprobación rápida: ¿ayudan las variables nuevas y el log?

Comparamos una Ridge con las variables originales frente a las nuevas, en CV de 5 folds.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from src.models import get_cv, evaluar_cv, ridge_log

prep_simple = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
])
base = Pipeline([("prep", prep_simple), ("model", Ridge())])
r_base, _ = evaluar_cv(base, X, y)
r_fe, _ = evaluar_cv(Pipeline([("prep", build_preprocessor()), ("model", Ridge())]), X_fe, y)
r_log, _ = evaluar_cv(ridge_log(), X_fe, y)
print(f"Ridge original:               {r_base.mean():.3f}")
print(f"Ridge + variables nuevas:     {r_fe.mean():.3f}")
print(f"Ridge + nuevas + log(price):  {r_log.mean():.3f}")

## 4. Conclusión

Las variables nuevas y modelar `log(price)` (con la corrección de *smearing* de `LogTargetRegressor`, ver `src/models.py`) bajan el RMSE. Este preprocesado es el que usan los notebooks 03 y 04.